# [파이썬 3일차 - 18. AI 분석 기초]

**작성자(author):** 장아현  
**작성일자(date):** 2026.03.29  
**변경내역(description):** 고객 리뷰 문장을 SentenceTransformer로 벡터 임베딩하고,  
PostgreSQL pgvector를 활용해 코사인 유사도 기반 유사 리뷰를 검색하는 시스템 구현

## 환경 설정

In [44]:
import os
from dotenv import load_dotenv

load_dotenv()

DB_HOST     = os.getenv("DB_HOST")
DB_NAME     = os.getenv("DB_NAME")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_PORT     = os.getenv("DB_PORT", "5432")

print(f"DB 연결 대상: {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

DB 연결 대상: reviewuser@localhost:5432/reviewdb


---
## 1단계 - SentenceTransformer로 리뷰 데이터 임베딩

In [45]:
from sentence_transformers import SentenceTransformer

# paraphrase-MiniLM-L6-v2 모델 로드 (384차원 벡터 출력)
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

reviews = [
    "배송이 빠르고 제품도 좋아요.",
    "품질이 기대 이상입니다!",
    "생각보다 배송이 오래 걸렸어요.",
    "배송은 느렸지만 포장은 안전했어요.",
    "아주 만족스러운 제품입니다."
]

# 리뷰 리스트를 384차원 벡터로 변환
embeddings = model.encode(reviews)

print(f"임베딩 완료: {len(embeddings)}개 리뷰, 벡터 차원: {embeddings.shape[1]}")
for review, emb in zip(reviews, embeddings):
    print(f"  - {review[:20]}... → shape={emb.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12957.42it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


임베딩 완료: 5개 리뷰, 벡터 차원: 384
  - 배송이 빠르고 제품도 좋아요.... → shape=(384,)
  - 품질이 기대 이상입니다!... → shape=(384,)
  - 생각보다 배송이 오래 걸렸어요.... → shape=(384,)
  - 배송은 느렸지만 포장은 안전했어요.... → shape=(384,)
  - 아주 만족스러운 제품입니다.... → shape=(384,)


---
## 2단계 - PostgreSQL 테이블 생성

In [46]:
import psycopg2

conn = psycopg2.connect(
    host=DB_HOST,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    port=DB_PORT
)
cur = conn.cursor()

# pgvector 확장 활성화 + 테이블 생성
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("""
    CREATE TABLE IF NOT EXISTS review_vectors (
        id        SERIAL PRIMARY KEY,
        review    TEXT,
        embedding VECTOR(384)
    );
""")
conn.commit()

print("테이블 생성 완료: review_vectors")

테이블 생성 완료: review_vectors


---
## 3단계 - 임베딩 벡터 저장

In [47]:
# 중복 저장 방지를 위해 기존 데이터 초기화
cur.execute("TRUNCATE TABLE review_vectors RESTART IDENTITY;")

for review, emb in zip(reviews, embeddings):
    emb_list = emb.tolist()
    cur.execute(
        "INSERT INTO review_vectors (review, embedding) VALUES (%s, %s)",
        (review, emb_list)
    )

conn.commit()
print(f"{len(reviews)}개 리뷰 벡터 저장 완료")

5개 리뷰 벡터 저장 완료


---
## 4단계 - 코사인 유사도 기반 유사 리뷰 검색

In [48]:
query = "배송이 느렸어요"
query_vec = model.encode([query])[0].tolist()

# <=> : pgvector 코사인 거리 연산자 (값이 작을수록 유사)
cur.execute(
    """
    SELECT review, embedding <=> %s::vector AS distance
    FROM review_vectors
    ORDER BY embedding <=> %s::vector
    LIMIT 3
    """,
    (query_vec, query_vec)
)

results = cur.fetchall()

print(f"쿼리: '{query}'")
print("=" * 45)
print(f"  {'순위':>4} | {'코사인 거리':>10} | 리뷰")
print(f"  {'-'*4}-+-{'-'*10}-+-{'-'*20}")
for rank, (review, distance) in enumerate(results, 1):
    print(f"  {rank:>4}위 | {distance:>10.4f} | {review}")

쿼리: '배송이 느렸어요'
    순위 |     코사인 거리 | 리뷰
  -----+------------+---------------------
     1위 |     0.0783 | 배송이 빠르고 제품도 좋아요.
     2위 |     0.0990 | 배송은 느렸지만 포장은 안전했어요.
     3위 |     0.1253 | 생각보다 배송이 오래 걸렸어요.


---
## 마무리 - DB 연결 종료

In [49]:
cur.close()
conn.close()
print("DB 연결 종료")

DB 연결 종료
